# Data and attacks

Four datasets carry the main results, CIFAR-10, CIFAR-100, GTSRB and Tiny ImageNet, with SVHN and EuroSAT added only where a clean-label attack needs a larger target class than the four main datasets can offer. Nine distinct triggers are in scope, described in full in `paper/sections/13-attacks.tex`: BadNets, Blend, SIG, WaNet, LF, Label-Consistent, BPP, Adaptive-Blend and TaCT. This notebook loads one dataset, builds every attack's trigger from the same registry the training code uses, and works through the poisoning protocol that decides which images an attack is even allowed to touch.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses

from dataclasses import asdict

from attacks import ATTACK_NAMES, build_attack, default_config
from data.registry import DATASET_REGISTRY
from evaluation.loaders import load_test_base
from attacks.poisoning import (
    AttackSuccessSet,
    attack_success_label,
    is_eval_poisonable,
    is_poisonable,
    poisoned_label,
)

DEVICE = torch.device("cpu")
DATASET = "cifar100"
TARGET_LABEL = 0
print("torch", torch.__version__, "device", DEVICE)


torch 2.6.0+cu124 device cpu


## The attack registry

`ATTACK_NAMES` carries 19 entries. An attack is a plain record of a name, a function that stamps a trigger, a label policy and a config, not a class hierarchy, so most of the extra entries are label-policy variants of a smaller set of triggers rather than new triggers. `badnet` is an alias for `badnet_a2o`, `generated` carries no default config because it reads somebody else's PNG triggers instead of stamping its own, and every `badnet_a2m*` entry stamps the identical checkerboard under an all-to-many label policy. Filtering those out leaves the 9 distinct triggers.

In [2]:
# badnet is an alias, generated has no default config, and every badnet_a2m*
# stamps the identical checkerboard under a different label policy.
SKIP = ("badnet", "generated")
TRIGGER_NAMES = [name for name in ATTACK_NAMES if name not in SKIP and "a2m" not in name]

spec = DATASET_REGISTRY[DATASET]
attacks = {
    name: build_attack(name, default_config(name), spec.image_size, TARGET_LABEL)
    for name in TRIGGER_NAMES
}

registry = pd.DataFrame(
    [
        {
            "attack": name,
            "label_mode": attacks[name].label_mode,
            "config": ", ".join(
                f"{key}={value}"
                for key, value in asdict(default_config(name)).items()
                if key != "label_mode"
            ),
        }
        for name in TRIGGER_NAMES
    ]
).set_index("attack")

skipped = [n for n in ATTACK_NAMES if n not in TRIGGER_NAMES]
print(f"registry entries {len(ATTACK_NAMES)}, distinct triggers {len(TRIGGER_NAMES)}")
print(f"skipped {len(skipped)}: {', '.join(skipped)}")
registry

registry entries 19, distinct triggers 10
skipped 9: badnet, badnet_a2m2, badnet_a2m4, badnet_a2m8, badnet_a2m16, badnet_a2m32, badnet_a2m64, badnet_a2m128, generated


,label_mode,config
attack,,
badnet_a2o,all_to_one,"patch_size=3, num_targets=None"
badnet_a2a,all_to_all,"patch_size=3, num_targets=None"
blend,all_to_one,"alpha=0.2, pattern_seed=0"
sig,clean_label,"amplitude=0.157, frequency=6.0, num_targets=1"
wanet,all_to_one,"control_grid_size=4, strength=0.5, field_seed=..."
lf,all_to_one,"strength=0.1, cutoff=4, pattern_seed=0"
lc,clean_label,"patch_size=3, adversarial_dir=, adversarial_ep..."
bpp,all_to_one,"bit_depth=3, dither=False, cover_rate=0.0"
adaptive_blend,all_to_one,"alpha=0.2, cover_rate=0.01, pattern_seed=0, ce..."


## What each trigger does to one image

BadNets, TaCT and the checkerboard corner of Label-Consistent all stamp the same 3 by 3 pattern into a fixed location, so their pixels are identical and any difference in their detection numbers comes from the label policy or the cover images, never from the trigger itself. Blend and Adaptive-Blend also share a pattern, but only at evaluation time. Adaptive-Blend's training trigger plants a random subset of the pattern per image, which is the mechanism behind why it survives detectors trained against the full pattern rather than an implementation shortcut.

In [3]:
test_base, _ = load_test_base(DATASET, "raw_data")
EXAMPLE_INDEX = 7
example_image, example_label = test_base[EXAMPLE_INDEX]

badnet_image = attacks["badnet_a2o"].apply_trigger(example_image, EXAMPLE_INDEX)
assert torch.equal(badnet_image, attacks["badnet_a2a"].apply_trigger(example_image, EXAMPLE_INDEX))
assert torch.equal(badnet_image, attacks["tact"].apply_trigger(example_image, EXAMPLE_INDEX))

blend_image = attacks["blend"].apply_trigger(example_image, EXAMPLE_INDEX)
assert torch.equal(blend_image, attacks["adaptive_blend"].apply_trigger_eval(example_image, EXAMPLE_INDEX))
assert not torch.equal(blend_image, attacks["adaptive_blend"].apply_trigger(example_image, EXAMPLE_INDEX))

print("badnet_a2o, badnet_a2a and tact stamp identical pixels")
print("blend and adaptive_blend stamp identical pixels at evaluation time only")
print(f"example image {tuple(example_image.shape)}, class {example_label}")

/lustre/home/pstika/projects/PSBD-ViT/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


badnet_a2o, badnet_a2a and tact stamp identical pixels
blend and adaptive_blend stamp identical pixels at evaluation time only
example image (3, 32, 32), class 14


In [4]:
def trigger_row_statistics(attack, image, index):
    """Max absolute change and touched-pixel fraction for one trigger on one image."""
    triggered = attack.apply_trigger(image, index)
    difference = (triggered - image).abs()
    statistics = {
        "triggered": triggered,
        "difference": difference,
        "max_abs": float(difference.max()),
        "touched_fraction": float((difference > 1e-6).float().mean()),
    }
    return statistics


def plot_trigger_grid(attacks, image, index):
    figure, axes = plt.subplots(len(attacks), 3, figsize=(6.4, 2.05 * len(attacks)))
    for axis_row, (name, attack) in zip(axes, attacks.items()):
        row = trigger_row_statistics(attack, image, index)
        amplified = row["difference"] / max(row["max_abs"], 1e-8)
        for axis, picture, title in zip(
            axis_row,
            [image, row["triggered"], amplified],
            [
                "clean",
                f"{name}  [{attack.label_mode}]",
                f"difference, x{1 / max(row['max_abs'], 1e-8):.0f}, "
                f"{100 * row['touched_fraction']:.1f}% of pixels touched",
            ],
        ):
            axis.imshow(picture.permute(1, 2, 0).numpy(), interpolation="nearest")
            axis.set_title(title, fontsize=8)
            axis.axis("off")
    figure.tight_layout()
    return figure


plot_trigger_grid(attacks, example_image, EXAMPLE_INDEX)
plt.show()

Patch triggers touch under 4 percent of the pixels and change them by a large amount. Blend, SIG and LF spread a small change over the whole image instead. This split matters later, because a whole-image trigger has no single group of tokens for an attention-routing story to point at, while a patch trigger does.

## The poisoning protocol

A trigger is only half of an attack. The other half is the label policy, and a sample is not always eligible for the same reason at training time and at evaluation time. For a dirty-label attack the two questions agree: an image is poisonable if it is not already the target, and the same condition asks whether the trigger fools it at test time. For a clean-label attack the two questions are opposites. Training can only poison an image that already carries the target label, since a clean-label attack never touches the label, while evaluation only makes sense on an image that does not, since the question being asked is whether the trigger fools a non-target image into the target class.

In [5]:
NUM_CLASSES = spec.num_classes
SAMPLE_LABELS = [0, 1, 42]

policy = pd.DataFrame(
    [
        {
            "label_mode": mode,
            "true_label": label,
            "train_eligible": is_poisonable(mode, label, TARGET_LABEL),
            "train_label": poisoned_label(mode, label, TARGET_LABEL, NUM_CLASSES),
            "eval_eligible": is_eval_poisonable(mode, label, TARGET_LABEL),
            "eval_target": attack_success_label(mode, label, TARGET_LABEL, NUM_CLASSES),
        }
        for mode in ("all_to_one", "all_to_all", "clean_label")
        for label in SAMPLE_LABELS
    ]
).set_index(["label_mode", "true_label"])

policy

train_eligible  train_label  eval_eligible  \
label_mode  true_label                                               
all_to_one  0                    False            0          False   
            1                     True            0           True   
            42                    True            0           True   
all_to_all  0                     True            1           True   
            1                     True            2           True   
            42                    True           43           True   
clean_label 0                     True            0          False   
            1                    False            1           True   
            42                   False           42           True   

                        eval_target  
label_mode  true_label               
all_to_one  0                     0  
            1                     0  
            42                    0  
all_to_all  0                     1  
            1                     2  
            42                   43  
clean_label 0                     0  
            1                     0  
            42                    0

Read the `clean_label` block against the other two. At the target label the training column says eligible and the evaluation column says not, and at every other label the reverse holds. `attacks.poisoning.AttackSuccessSet` is the single class both the training entrypoint and the checkpoint-metadata reader use to build the correct pool for each question, so a clean-label evaluation set is never accidentally the training pool.

In [6]:
import torchvision.transforms.v2 as transforms_v2

from data.loading import extract_labels

test_labels = extract_labels(test_base)
sig_attack = attacks["sig"]
normalize = transforms_v2.Normalize(mean=spec.mean, std=spec.std)
success_set = AttackSuccessSet(test_base, test_labels, sig_attack, normalize, NUM_CLASSES)

served_targets = {int(success_set[position][1]) for position in range(0, len(success_set), 97)}
served_true_labels = {test_labels[i] for i in success_set.indices}

print(f"clean-label evaluation set size {len(success_set)} of {len(test_labels)} test images")
print(f"true classes it draws from       {len(served_true_labels)} of {NUM_CLASSES}, excludes the target: "
      f"{TARGET_LABEL not in served_true_labels}")
print(f"labels the trigger is asked to reach {served_targets}")
assert served_targets == {TARGET_LABEL}

clean-label evaluation set size 9900 of 10000 test images
true classes it draws from       99 of 100, excludes the target: True
labels the trigger is asked to reach {0}


## Poison-rate mechanics

A requested poison rate is read against the whole training set, but the images an attack is allowed to draw from can be far smaller than that. `attacks.poisoning.choose_poison_indices` draws without replacement from the eligible pool and clamps silently once the request exceeds it, so a requested rate above the pool's share trains the identical index set as the cap itself.

In [7]:
from attacks.poisoning import choose_poison_indices
from data.loading import base_image_transform, load_clean_datasets
from data.splits import PSBD_SPLIT_SEED

RATES = (0.01, 0.05, 0.10)
MODE_ATTACKS = {"all_to_one": "badnet_a2o", "all_to_all": "badnet_a2a", "clean_label": "sig"}

train_set, _ = load_clean_datasets(DATASET, base_image_transform(spec.image_size), "raw_data")
train_labels = extract_labels(train_set)

rate_rows = []
for label_mode, attack_name in MODE_ATTACKS.items():
    attack = build_attack(attack_name, default_config(attack_name), spec.image_size, TARGET_LABEL)
    for rate in RATES:
        chosen = choose_poison_indices(train_labels, attack, rate, seed=PSBD_SPLIT_SEED)
        requested_count = int(round(rate * len(train_labels)))
        rate_rows.append(
            {
                "label_mode": label_mode,
                "requested_rate": rate,
                "requested_count": requested_count,
                "realized_count": len(chosen),
                "realized_rate": len(chosen) / len(train_labels),
                "capped": len(chosen) < requested_count,
            }
        )

pd.DataFrame(rate_rows).set_index(["label_mode", "requested_rate"]).round(5)

/lustre/home/pstika/projects/PSBD-ViT/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


requested_count  realized_count  realized_rate  \
label_mode  requested_rate                                                   
all_to_one  0.01                        500             500           0.01   
            0.05                       2500            2500           0.05   
            0.10                       5000            5000           0.10   
all_to_all  0.01                        500             500           0.01   
            0.05                       2500            2500           0.05   
            0.10                       5000            5000           0.10   
clean_label 0.01                        500             500           0.01   
            0.05                       2500             500           0.01   
            0.10                       5000             500           0.01   

                            capped  
label_mode  requested_rate          
all_to_one  0.01             False  
            0.05             False  
            0.10             False  
all_to_all  0.01             False  
            0.05             False  
            0.10             False  
clean_label 0.01             False  
            0.05              True  
            0.10              True

`all_to_one` and `all_to_all` read back the rate they were asked for, because a dirty-label pool is almost the whole training set. `clean_label` caps at 1 percent on CIFAR-100, because its pool is the target class alone and that class holds 1 percent of the training images. The cap is arithmetic, not a bug, and it is why the paper trains clean-label attacks at whichever target class gives the largest reachable rate: class 0 on CIFAR-10, CIFAR-100 and Tiny ImageNet, class 1 on GTSRB, where class 0 alone would cap SIG and Label-Consistent at a rate too small to implant.

## What this notebook establishes

9 distinct triggers, 3 label policies with an asymmetric eligibility rule for the clean-label case, and a poison-rate cap that is set by class balance rather than by the attack's own code. `paper/sections/13-attacks.tex` carries the exact parameters and the attack success rate each trigger reaches at each rate, on every dataset.